# Проект 14: Итеративный самокорректирующийся агент

**Граф:** Писатель → Критик → Редактор → (цикл или финиш)

**Фреймворк:** LangGraph + GPT-4o-mini

## 1. Установка зависимостей

In [1]:
!pip install langgraph langchain-openai python-dotenv

  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/543.9 kB ? eta -:--:--
   ---------------------------------------- 543.9/543.9 kB 3.8 MB/s  0:00:00
Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
   ---------------------------------------- 0.0/921.1 kB ? eta -:--:--
   ---------------------------------------- 921.1/921.1 kB 4.6 MB/s  0:00:00
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)

   ----------------------------------------  0/23 [zstandard]
   - --------------------------------------  1/23 [xxhash]
   ----- ----------------------------------  3/23 [urllib3]
   ----- ----------------------------------  3/23 [urllib3]
   ----- ----------------------------------  3/23 [urllib3]
   ----- ----------------------------------  3/23 [urllib3]
   ------ ---------------------------------  4/23 [tenacity]
   -------- 


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os, json
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
print("Готово")

c:\Users\dinam\OneDrive\Рабочий стол\homeworks\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Готово


## 2. Состояние графа (State)

Единая структура данных, которая передаётся между всеми узлами.

In [3]:
class AgentState(TypedDict):
    task: str                    # Исходное задание пользователя
    content: str                 # Текущая версия текста/кода
    critique: str                # Замечания критика
    approved: bool               # Прошёл ли контент проверку
    iteration: int               # Номер текущей итерации
    max_iterations: int          # Максимум итераций (защита от бесконечного цикла)
    history: list                # Лог всех версий для анализа

## 3. Узлы графа

In [4]:
# ─────────────────────────────────────────────
# УЗЕЛ 1: ПИСАТЕЛЬ
# Генерирует первичный контент по заданию.
# ─────────────────────────────────────────────
def writer_node(state: AgentState) -> AgentState:
    iteration = state["iteration"]
    print(f"\n{'='*55}")
    print(f"✍️  ПИСАТЕЛЬ — итерация #{iteration + 1}")
    print(f"{'='*55}")

    if iteration == 0:
        # Первый черновик
        prompt = f"""Выполни следующее задание. Напиши только результат, без пояснений.

Задание: {state['task']}"""
    else:
        # Редактирование на основе критики
        prompt = f"""Тебе нужно улучшить текст на основе критики.

Задание: {state['task']}

Текущая версия:
{state['content']}

Критика (что нужно исправить):
{state['critique']}

Напиши улучшенную версию. Только результат, без пояснений."""

    response = llm.invoke([
        SystemMessage(content="Ты — профессиональный автор. Пишешь чётко и качественно."),
        HumanMessage(content=prompt)
    ])

    new_content = response.content
    print(f"\n📄 Версия #{iteration + 1}:")
    print(new_content)

    # Обновляем историю
    history = state["history"] + [{
        "iteration": iteration + 1,
        "version": new_content,
        "critique": state.get("critique", "")
    }]

    return {
        **state,
        "content": new_content,
        "iteration": iteration + 1,
        "approved": False,
        "history": history,
    }

In [5]:
# ─────────────────────────────────────────────
# УЗЕЛ 2: КРИТИК
# Анализирует контент и выносит вердикт.
# Возвращает JSON с полями: approved, critique, score.
# ─────────────────────────────────────────────
CRITIQUE_CRITERIA = """
Оценивай по следующим критериям:
1. Точность — нет фактических ошибок
2. Полнота — задание выполнено полностью
3. Ясность — текст понятен и структурирован
4. Стиль — профессиональный тон, нет воды
5. Грамотность — нет грамматических ошибок
"""

def critic_node(state: AgentState) -> AgentState:
    print(f"\n{'='*55}")
    print(f"🔍 КРИТИК — итерация #{state['iteration']}")
    print(f"{'='*55}")

    prompt = f"""Проверь следующий текст на соответствие заданию.

Задание: {state['task']}

Текст для проверки:
{state['content']}

Критерии оценки:{CRITIQUE_CRITERIA}

Ответь СТРОГО в формате JSON (без markdown-блоков):
{{
  "approved": true/false,
  "score": <число от 1 до 10>,
  "critique": "<конкретные замечания через точку с запятой, или 'Всё отлично' если approved=true>",
  "strengths": "<что сделано хорошо>"
}}

approved=true только если score >= 8 и нет существенных замечаний."""

    response = llm.invoke([
        SystemMessage(content="Ты — строгий редактор. Оцениваешь объективно, указываешь конкретные проблемы."),
        HumanMessage(content=prompt)
    ])

    raw = response.content.strip()
    # Убираем markdown-обёртку если есть
    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        # Фолбэк если LLM не выдал чистый JSON
        result = {"approved": False, "score": 5, "critique": raw, "strengths": ""}

    approved = result.get("approved", False)
    score    = result.get("score", 0)
    critique = result.get("critique", "")
    strength = result.get("strengths", "")

    verdict = "✅ ОДОБРЕНО" if approved else "❌ ТРЕБУЕТ ПРАВОК"
    print(f"\n{verdict} | Оценка: {score}/10")
    print(f"💪 Плюсы: {strength}")
    print(f"📝 Замечания: {critique}")

    # Если достигнут лимит итераций — принудительно одобряем
    if state["iteration"] >= state["max_iterations"]:
        print(f"\n⚠️  Достигнут лимит итераций ({state['max_iterations']}). Завершаем.")
        approved = True

    return {
        **state,
        "approved": approved,
        "critique": critique,
    }

In [6]:
# ─────────────────────────────────────────────
# УСЛОВНЫЙ ПЕРЕХОД
# Определяет: идти на финиш или на новый круг.
# ─────────────────────────────────────────────
def should_continue(state: AgentState) -> str:
    if state["approved"]:
        print("\n🏁 Переход: ФИНИШ")
        return "finish"
    else:
        print("\n🔄 Переход: НОВАЯ ИТЕРАЦИЯ")
        return "revise"

## 4. Сборка графа LangGraph

In [7]:
# Инициализация графа
graph = StateGraph(AgentState)

# Регистрация узлов
graph.add_node("writer", writer_node)
graph.add_node("critic",  critic_node)

# Рёбра
graph.set_entry_point("writer")        # Старт — всегда Писатель
graph.add_edge("writer", "critic")     # После написания — всегда Критик

# Условный переход из Критика
graph.add_conditional_edges(
    "critic",
    should_continue,
    {
        "finish": END,       # Одобрено → конец
        "revise": "writer",  # Не одобрено → снова Писатель
    }
)

# Компиляция
app = graph.compile()
print("Граф скомпилирован")
print("\nСхема: writer → critic → (finish | writer → critic → ...)")

Граф скомпилирован

Схема: writer → critic → (finish | writer → critic → ...)


## 5. Запуск агента

In [8]:
def run_agent(task: str, max_iterations: int = 4) -> AgentState:
    """Запускает самокорректирующийся агент и возвращает финальное состояние."""

    initial_state: AgentState = {
        "task": task,
        "content": "",
        "critique": "",
        "approved": False,
        "iteration": 0,
        "max_iterations": max_iterations,
        "history": [],
    }

    final_state = app.invoke(initial_state)

    print(f"\n{'='*55}")
    print("🎯 ФИНАЛЬНЫЙ РЕЗУЛЬТАТ")
    print(f"{'='*55}")
    print(f"Итераций пройдено: {final_state['iteration']}")
    print(f"\n{final_state['content']}")

    return final_state

## 6. Тест 1 — Написание текста
**Задание:** Написать профессиональный абзац о важности резервного копирования данных.

In [9]:
result1 = run_agent(
    task=(
        "Напиши профессиональный абзац (4–6 предложений) о важности резервного копирования данных. "
        "Текст должен быть убедительным, содержать конкретный факт или статистику, "
        "и заканчиваться призывом к действию."
    ),
    max_iterations=4,
)


✍️  ПИСАТЕЛЬ — итерация #1

📄 Версия #1:
Резервное копирование данных является критически важной практикой для обеспечения безопасности информации в современном цифровом мире. По данным статистики, более 60% компаний, потерявших данные, закрываются в течение шести месяцев после инцидента. Это подчеркивает необходимость регулярного создания резервных копий, чтобы избежать потерь, которые могут привести к финансовым и репутационным убыткам. Не откладывайте на завтра то, что можно сделать сегодня: начните резервировать свои данные прямо сейчас, чтобы защитить себя и свой бизнес от непредвиденных ситуаций.

🔍 КРИТИК — итерация #1

❌ ТРЕБУЕТ ПРАВОК | Оценка: 7/10
💪 Плюсы: Текст хорошо структурирован, содержит важную информацию о последствиях потери данных и призыв к действию.
📝 Замечания: Статистика не указана конкретно, нужно уточнить источник; фраза 'не откладывайте на завтра' может звучать менее профессионально; стоит добавить больше конкретики о методах резервного копирования.

🔄 Перех

## 7. Тест 2 — Генерация кода
**Задание:** Написать Python-функцию с документацией и обработкой ошибок.

In [10]:
result2 = run_agent(
    task=(
        "Напиши Python-функцию `flatten(nested_list)`, которая рекурсивно "
        "разворачивает вложенный список любой глубины в плоский. "
        "Требования: docstring, аннотации типов, обработка случая когда входной "
        "аргумент не является списком, пример использования в комментарии."
    ),
    max_iterations=4,
)


✍️  ПИСАТЕЛЬ — итерация #1

📄 Версия #1:
```python
from typing import Any, List

def flatten(nested_list: Any) -> List[Any]:
    """
    Рекурсивно разворачивает вложенный список любой глубины в плоский.

    :param nested_list: Вложенный список для разворачивания.
    :return: Плоский список.
    """
    if not isinstance(nested_list, list):
        raise ValueError("Входной аргумент должен быть списком.")
    
    flat_list = []
    for item in nested_list:
        if isinstance(item, list):
            flat_list.extend(flatten(item))
        else:
            flat_list.append(item)
    return flat_list

# Пример использования:
# result = flatten([1, [2, [3, 4], 5], 6])
# print(result)  # Вывод: [1, 2, 3, 4, 5, 6]
```

🔍 КРИТИК — итерация #1

✅ ОДОБРЕНО | Оценка: 9/10
💪 Плюсы: Функция корректно реализует требуемую логику, docstring содержит необходимую информацию, аннотации типов присутствуют, обработка некорректного входного аргумента выполнена правильно, пример использования понят

## 8. История итераций

In [11]:
print("ИСТОРИЯ ИТЕРАЦИЙ — Тест 1 (текст)")
print("=" * 55)
for entry in result1["history"]:
    print(f"\n--- Итерация #{entry['iteration']} ---")
    if entry["critique"]:
        print(f"Критика предыдущей версии: {entry['critique']}")
    print(f"Версия:\n{entry['version'][:300]}{'...' if len(entry['version']) > 300 else ''}")

ИСТОРИЯ ИТЕРАЦИЙ — Тест 1 (текст)

--- Итерация #1 ---
Версия:
Резервное копирование данных является критически важной практикой для обеспечения безопасности информации в современном цифровом мире. По данным статистики, более 60% компаний, потерявших данные, закрываются в течение шести месяцев после инцидента. Это подчеркивает необходимость регулярного создания...

--- Итерация #2 ---
Критика предыдущей версии: Статистика не указана конкретно, нужно уточнить источник; фраза 'не откладывайте на завтра' может звучать менее профессионально; стоит добавить больше конкретики о методах резервного копирования.
Версия:
Резервное копирование данных является критически важной практикой для обеспечения безопасности информации в современном цифровом мире. Согласно исследованию компании Gartner, около 60% организаций, потерявших данные, закрываются в течение шести месяцев после инцидента. Это подчеркивает необходимость...
